# Data Mining 1: Raw Job Data Cleaning & Preprocessing

This notebook demonstrates the preprocessing and cleaning pipeline for raw tech job postings (Naukri.com dataset):
- Column pruning (dropping redundant metadata fields)
- Deduplication and missing value imputation
- Regex-based extraction of salary ranges and converting to numeric figures
- Experience level extraction and IT keyword filtering
- Export of cleaned subset for downstream modeling

In [8]:
# Importing Pandas To Manipulate Dataset

import pandas as pd

# Reading The Dataset
df = pd.read_csv("Datasets/naukri_jobs_full_dataset.csv")

# Remove Completely Useless Columns
drop_cols = [
    "logoPath","logoPathV3","footerPlaceholderLabel","footerPlaceholderColor",
    "isSaved","jdURL","groupId","isTopGroup","companyJobsUrl","consultant",
    "hideClientName","clientLogo","brandingTags","exclusive","clientHeadline",
    "clientCareersUrl","clientGroupId","smbJobFields","internshipTags"
]

df = df.drop(columns=drop_cols, errors="ignore")

# Checking Data General Information
df.info()

/var/folders/_6/1m3sr4td30148nvmb70jwp6w0000gn/T/ipykernel_31386/716766858.py:6: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Datasets/naukri_jobs_full_dataset.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103657 entries, 0 to 103656
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   title              103657 non-null  object 
 1   jobId              103657 non-null  int64  
 2   currency           103657 non-null  object 
 3   companyName        103657 non-null  object 
 4   tagsAndSkills      102620 non-null  object 
 5   companyId          103657 non-null  int64  
 6   ambitionBoxData    73903 non-null   object 
 7   jobDescription     103657 non-null  object 
 8   showMultipleApply  103657 non-null  bool   
 9   createdDate        103657 non-null  object 
 10  mode               103657 non-null  object 
 11  board              103657 non-null  int64  
 12  vacancy            5612 non-null    float64
 13  experience         102610 non-null  object 
 14  salary             103657 non-null  object 
 15  location           103654 non-null  object 
 16  hi

In [11]:
# Remove Duplicate Jobs

df = df.drop_duplicates()

# Handle Missing Values

df = df.dropna(subset=["salary"])
df["tagsAndSkills"] = df["tagsAndSkills"].fillna("Unknown")
df["jobType"] = df["jobType"].fillna("Unknown")

In [13]:
# Clean Salary Column

df = df[~df["salary"].str.contains("Not", case=False, na=False)]
import re

def extract_salary(x):
    nums = re.findall(r'\d+', str(x))
    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2
    elif len(nums) == 1:
        return int(nums[0])
    else:
        return None

df["salary_clean"] = df["salary"].apply(extract_salary)

In [15]:
# Clean Experience Column

def clean_exp(x):
    nums = re.findall(r'\d+', str(x))
    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2
    elif len(nums) == 1:
        return int(nums[0])
    return None

df["experience_clean"] = df["experience"].apply(clean_exp)

# Keep Only IT Jobs

keywords = "data|software|engineer|developer|analyst|ai|machine learning|cloud"

df = df[df["title"].str.contains(keywords, case=False, na=False)]

In [17]:
# Remove Remaining Missing Data

df = df.dropna(subset=["salary_clean","experience_clean"])

# Save Clean Dataset

df.to_csv("clean_it_salary_dataset.csv", index=False)

print("Final Rows:", len(df))

Final Rows: 4592


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4592 entries, 0 to 103654
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              4592 non-null   object 
 1   jobId              4592 non-null   int64  
 2   currency           4592 non-null   object 
 3   companyName        4592 non-null   object 
 4   tagsAndSkills      4592 non-null   object 
 5   companyId          4592 non-null   int64  
 6   ambitionBoxData    1864 non-null   object 
 7   jobDescription     4592 non-null   object 
 8   showMultipleApply  4592 non-null   bool   
 9   createdDate        4592 non-null   object 
 10  mode               4592 non-null   object 
 11  board              4592 non-null   int64  
 12  vacancy            1855 non-null   float64
 13  experience         4592 non-null   object 
 14  salary             4592 non-null   object 
 15  location           4592 non-null   object 
 16  hiringFor          989 non-